# Data preparation

Load the supplied data, normalize its text, remove duplicate and conflicting training examples, and save the datasets used for modelling.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import clean_dataframe

In [ ]:
columns = ["tweet_id", "entity", "sentiment", "text"]
train_df = pd.read_csv(PROJECT_ROOT / "data/twitter_training.csv", header=None, names=columns)
validation_df = pd.read_csv(PROJECT_ROOT / "data/twitter_validation.csv", header=None, names=columns)

train_clean = clean_dataframe(train_df)
validation_clean = clean_dataframe(validation_df)

print(f"Training rows: {len(train_df):,} -> {len(train_clean):,}")
print(f"Validation rows: {len(validation_df):,} -> {len(validation_clean):,}")

In [ ]:
train_clean = train_clean.drop_duplicates(
    subset=["entity", "sentiment", "text"]
).reset_index(drop=True)

label_counts = train_clean.groupby(["entity", "text"])["sentiment"].nunique()
conflicting_pairs = set(label_counts[label_counts > 1].index)
conflict_mask = train_clean.apply(
    lambda row: (row["entity"], row["text"]) in conflicting_pairs,
    axis=1,
)
train_clean = train_clean.loc[~conflict_mask].reset_index(drop=True)

print(f"Conflicting entity-text pairs removed: {len(conflicting_pairs):,}")
print(f"Final training rows: {len(train_clean):,}")

In [ ]:
output_dir = PROJECT_ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
train_clean.to_csv(output_dir / "train_clean.csv", index=False)
validation_clean.to_csv(output_dir / "validation_clean.csv", index=False)